# 멀티모달 RAG Colab 실습
- LangChain + ChromaDB + GPT-4o-mini
- 이미지 정보를 반영하는 멀티모달 RAG 예제

## 0.환경 준비

In [1]:
# 필수 패키지 설치
!pip install --upgrade langchain langchain-openai langchain-chroma chromadb pillow

In [2]:
# 구글 드라이브 마운트 및 경로
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1.환경 변수 및 경로 설정
- Colab **Secrets(userdata)** 에서 API 키를 불러와 환경변수로 세팅

In [3]:
from google.colab import userdata
import os

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

BASE_PATH = '/content/drive/MyDrive'

## 2.이미지 변환
- 이미지 파일을 Base64 문자열로 바꿈
- Why? LLM(비전) API에 이미지를 "텍스트처럼" 전달하기 위해서
- 작은 썸네일만 저장하거나 경로만 저장하면 DB가 가벼워짐

In [4]:
import base64
from PIL import Image
from io import BytesIO
from openai import OpenAI

In [5]:
# 이미지 파일 -> BASE64 인코딩
def image_to_base64(image_path):
    with open(image_path, 'rb') as f:
        encoded = base64.b64encode(f.read()).decode('utf-8')
    return encoded

In [6]:
# GPT-4o 기반 이미지 요약(OpenAI Vision)
client = OpenAI()

In [7]:
def summarize_image_with_gpt4o(image_path):
    '''
    단일 이미지를 안전교육/화재대피 관점으로 한국어 요약합니다.
    - 입력 : 로컬 이미지 경로
    - 처리 : 이미지를 BASE64로 인코딩하여 data URL로 vision 모델에 전달
    - 출력 : 한국어 요약 텍스(최대 토큰 256)
    '''
    with open(image_path, "rb") as f:
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": "아래 이미지를 안전교육 및 화재대피 관점에서 주제 문구와 각 장면에 대해 주요 정보에 대한 손실없이 한국어로 요약해주세요."},
                {"role": "user", "content": [{"type": "image_url", "image_url": {"url": "data:image/jpeg;base64," + base64.b64encode(f.read()).decode()}}]}
            ],
            max_tokens=256,
        )
    # 응답 객체에서 요약텍스트만 출력
    return response.choices[0].message.content

## 3.벡터DB 구축 & 데이터 적재
- 임베딩은 텍스트만(요약문) 대상
- 이미지는 메타데이터로만 저장되어, 검색은 텍스트 기준으로 이뤄짐

In [8]:
import os
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

In [9]:
persist_directory = f"{BASE_PATH}/chroma"

- 벡터DB 생성

In [10]:
embedding = OpenAIEmbeddings(model='text-embedding-3-small')
vectordb = Chroma(
    collection_name='multimodal_rag',       # 벡터DB 안에서 사용할 컬렉션 이름
    embedding_function=embedding,           # 텍스트를 벡터로 바꿀 때 사용할 임베딩 함수
    persist_directory=persist_directory
)

- 이미지 적재

In [11]:
import os

# '이미지->요약->벡터db 저장' 과정을 자동으로 반복하는 단계
# 폴더 안의 모든 이미지를 한 장씩 텍스트 요약으로 바꿔서 데이터베이스에 넣는 과정

figures_dir = os.path.join(BASE_PATH, 'figures')

# figures 디렉토리가 없으면 생성
os.makedirs(figures_dir, exist_ok=True)

image_files = [
    os.path.join(figures_dir, fname)
    for fname in os.listdir(figures_dir)
    if fname.lower().endswith('.jpg')
]
image_files = sorted(image_files)

for img_path in image_files:
    b64_str = image_to_base64(img_path)
    summary = summarize_image_with_gpt4o(img_path)
    vectordb.add_texts(
        texts=[summary],
        metadatas=[{"base64_image": b64_str, "image_path": img_path}]
    )

## 4.데이터 확인

In [12]:
# vectordb._collection.get()으로 데이터 전체 확인
docs = vectordb._collection.get()

# docs는 dict 구조이며, 'ids', 'documents', 'metadatas' 등이 key로 포함됩니다.
for idx, (doc, meta) in enumerate(zip(docs['documents'], docs['metadatas'])):
    print(f"[{idx+1}]")
    print("요약텍스트:", doc)
    print("이미지경로:", meta['base64_image'])
    print("BASE64 길이:", len(meta['base64_image']))
    print("-" * 40)


In [13]:
import IPython.display as display

def show_base64_image(b64_str):
    img = Image.open(BytesIO(base64.b64decode(b64_str)))
    display.display(img)

# 사용자 질문을 받아 벡터DB에서 관련 문서를 검색하고
# 검색된 요약문을 원본 이지를 함께 보여주는 함수
def search_and_show(query, top_k=2):
    docs = vectordb.similarity_search(query, k=top_k)
    for i, doc in enumerate(docs):
        print(f"\n[TOP-{i+1}] 요약: {doc.page_content}")
        show_base64_image(doc.metadata['base64_image'])


## 5.LCEL RAG 체인 구성
- 질문 → 텍스트 임베딩 검색(k=1) → 찾은 요약 텍스트만으로 답
- 모델은 비용/속도 고려해 gpt-4o-mini 사용
- k=1~3로 실험하면서 근거의 다양성과 노이즈 밸런스를 체크

In [14]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

In [15]:
llm = ChatOpenAI(
    model='gpt-4o',
    temperature=0
)
retriever = vectordb.as_retriever(search_kwargs={"k":1})
rag_prompt = ChatPromptTemplate.from_template("""
Answer in Korean using only the context below.
If the answer is not in the context, say that you do not know.

[Context]
{context}

[Question]
{question}
""")

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [16]:
# OpenAI API 연결 테스트
try:
    test_response = llm.invoke("안녕하세요, 잘 작동하는지 테스트합니다.")
    print("OpenAI API 연결 성공!")
    print("테스트 응답:", test_response.content)
except Exception as e:
    print(f"OpenAI API 연결 실패: {e}")

OpenAI API 연결 성공!
테스트 응답: 안녕하세요! 잘 작동하고 있습니다. 무엇을 도와드릴까요?


In [17]:
from openai import OpenAI

try:
    client = OpenAI()
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",  # Using a common model for a quick test
        messages=[{"role": "user", "content": "hello"}],
        max_tokens=5
    )
    print("OpenAI API connection successful!")
    print("Test response:", response.choices[0].message.content)
except Exception as e:
    print(f"OpenAI API connection failed: {e}")
    print("Please ensure your OPENAI_API_KEY is correctly set in Colab Secrets and has the necessary permissions. You may also need to restart the runtime after updating the key.")

OpenAI API connection successful!
Test response: Hello! How can I


In [18]:
qa_chain = rag_prompt | llm | StrOutputParser()

## 6.질의/결과
- 답변과 함께 근거 이미지/텍스트를 보여주는 RAG Explainability 구현

In [19]:
# Example 1
query = "화재 시 피난 요령을 설명해줘."
source_docs = retriever.invoke(query)

result = {
    "result": qa_chain.invoke({
        "context":format_docs(source_docs),
        "question":query
    }),
    "source_documents": source_docs
}
print(result['result'])

모르겠습니다.


In [20]:
for doc in result['source_documents']:
    show_base64_image(doc.metadata['base64_image'])
    print(doc.page_content)

In [21]:
# Example 2
query2 = "119 신고 및 심폐소생술 절차를 알려줘."
source_docs2 = retriever.invoke(query2)

result2 = {
    "result": qa_chain.invoke({
        "context":format_docs(source_docs2),
        "question":query2
    }),
    "source_documents": source_docs2
}
print(result2['result'])


모르겠습니다.


In [22]:
for doc in result2['source_documents']:
    show_base64_image(doc.metadata['base64_image'])
    print(doc.page_content)

In [23]:
import os

# Check if the OPENAI_API_KEY is set in the environment variables
if 'OPENAI_API_KEY' in os.environ:
    # To avoid exposing the full key, print only the beginning and end
    key_value = os.environ['OPENAI_API_KEY']
    print(f"OPENAI_API_KEY is loaded. (Starts with: {key_value[:5]}..., Ends with: ...{key_value[-5:]})")
else:
    print("OPENAI_API_KEY is NOT set in the environment variables.")

OPENAI_API_KEY is loaded. (Starts with: sk-pr..., Ends with: ...vx40A)


## [실습]
1. 이미지 요약 프롬프트 개선 실험. 같은 이미지에 대해 자유 요약 프롬프트와 구조화 요약 프롬프트를 각각 적용하고, 검색 결과 품질을 비교하기.
2. 이미지 요약 형식 구조화. 이미지 요약을 [주제], [상황], [위험요소], [행동요령], [키워드] 형식으로 저장하기.
3. top_k 값에 따른 검색 결과 비교. `k=1`, `k=2`, `k=3`으로 바꿔가며 같은 질문에 대해 검색되는 문서와 최종 답변이 어떻게 달라지는지 분석하기.
4. 이미지 메타데이터 추가 저장하기. 이미지 저장 시 `image_path`, `category`, `title`, `created_at` 같은 메타데이터를 함께 저장하고, 검색 결과에서 메타데이터까지 출력하기.
5. 나만의 이미지 RAG 챗봇 만들기. 안전교육, 응급처치, 교통안전, 제품 매뉴얼 중 하나를 주제로 이미지 5장 이상을 수집하고, 이미지 요약 → 벡터DB 저장 → 질문 답변까지 구현하기.